# 01 · Exploratory data analysis: ShanghaiTech Campus

What the data looks like before any model: how many videos and frames per camera, how much of the test set is abnormal, and what the 13 cameras see. Figures are saved to `reports/figures/eda_*.png`.

Run: `jupyter nbconvert --to notebook --execute --inplace notebooks/01_eda.ipynb` (needs the prepared data: `bash run_all.sh` or `dvc pull`).

In [ ]:
import json
import os
import sys
from pathlib import Path
if Path.cwd().name == "notebooks":          # run from the project root, wherever Jupyter started
    os.chdir("..")
sys.path.insert(0, ".")
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from src.utils import frames_dir, labels_dir, list_frames, load_config

cfg = load_config()
FIG = Path("reports/figures"); FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3})
split = json.loads((Path(cfg["paths"]["metadata"]) / "split.json").read_text())

## 1. Data card and split

In [ ]:
card = json.loads((Path(cfg["paths"]["metadata"]) / "shanghaitech.json").read_text())
print(f"{card['videos']} videos, {card['frames']:,} frames at {card['fps']} fps; problems found: {card['problems'] or 'none'}")
pd.DataFrame(card["official_split"]).T

## 2. Videos and frames per camera
A video name is `<camera>_<number>`: training videos have 3 digits (`01_001`), test videos 4 (`01_0014`).

In [ ]:
root = frames_dir(cfg, "shanghaitech")
rows = [{"video": v, "split": s, "camera": v.split("_")[0], "frames": len(list_frames(root / v))}
        for s in ("train", "test") for v in split[s]]
videos = pd.DataFrame(rows)
per_cam = videos.pivot_table(index="camera", columns="split", values="frames", aggfunc=["count", "sum"]).fillna(0).astype(int)
per_cam.columns = [f"{a} {b}" for a, b in per_cam.columns]
per_cam = per_cam.rename(columns={"count test": "test videos", "count train": "train videos",
                                  "sum test": "test frames", "sum train": "train frames"})
per_cam

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
per_cam[["train videos", "test videos"]].plot.bar(ax=ax[0], color=["#2a78d6", "#eb6834"], width=0.8)
ax[0].set_title("Videos per camera"); ax[0].set_xlabel("camera"); ax[0].set_ylabel("videos")
for s, c in (("train", "#2a78d6"), ("test", "#eb6834")):
    ax[1].hist(videos.loc[videos.split == s, "frames"], bins=30, alpha=0.7, color=c, label=s)
ax[1].set_title("Video length"); ax[1].set_xlabel("frames per video"); ax[1].set_ylabel("videos"); ax[1].legend()
plt.tight_layout(); plt.savefig(FIG / "eda_videos_per_camera.png", bbox_inches="tight"); plt.show()

## 3. How much of the test set is abnormal
Ground truth = the dataset's own per-frame labels (`test_frame_mask`). Training videos are all normal (unsupervised setting).

In [ ]:
lab = {v: np.load(labels_dir(cfg) / f"{v}.npy").astype(int).ravel() for v in split["test"]}
test = videos[videos.split == "test"].copy()
test["abnormal frames"] = test.video.map(lambda v: int(lab[v].sum()))
test["abnormal share"] = test["abnormal frames"] / test.video.map(lambda v: len(lab[v]))
test["events"] = test.video.map(lambda v: int(((np.diff(np.r_[0, lab[v]]) == 1)).sum()))
y = np.concatenate(list(lab.values()))
print(f"test: {len(y):,} frames, {int(y.sum()):,} abnormal ({y.mean():.1%}); "
      f"{(test['abnormal frames'] == 0).sum()} test videos without any abnormal frame; "
      f"{test.events.sum()} abnormal events (runs of abnormal frames)")
test.groupby("camera")[["abnormal frames", "abnormal share"]].agg({"abnormal frames": "sum", "abnormal share": "mean"}).round(3).T

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
cam = test.groupby("camera")["abnormal share"].mean()
ax[0].bar(cam.index, cam.values, color="#eb6834"); ax[0].set_ylim(0, 1)
ax[0].set_title("Abnormal share of the test frames, per camera"); ax[0].set_xlabel("camera"); ax[0].set_ylabel("share of frames")
ax[1].hist(test["abnormal share"], bins=20, color="#eb6834", alpha=0.85)
ax[1].set_title("Abnormal share per test video"); ax[1].set_xlabel("share of abnormal frames"); ax[1].set_ylabel("videos")
plt.tight_layout(); plt.savefig(FIG / "eda_abnormal_share.png", bbox_inches="tight"); plt.show()

## 4. When anomalies happen
Each row is one test video (first 25), each column a frame: orange = abnormal. Anomalies are long runs (a person running, a bike), which is why smoothing the score over time helps (model 6).

In [ ]:
sel = split["test"][:25]
n = max(len(lab[v]) for v in sel)
img = np.full((len(sel), n), np.nan)
for i, v in enumerate(sel):
    img[i, :len(lab[v])] = lab[v]
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.imshow(img, aspect="auto", cmap=plt.matplotlib.colors.ListedColormap(["#dfe6ee", "#eb6834"]), interpolation="nearest")
ax.set_yticks(range(len(sel)), sel, fontsize=7); ax.set_xlabel("frame"); ax.grid(False)
ax.set_title("Ground truth over time (orange = abnormal)")
plt.tight_layout(); plt.savefig(FIG / "eda_label_timeline.png", bbox_inches="tight"); plt.show()

## 5. What the cameras see
One frame per camera (training videos). Different viewpoints, distances and crowd densities: one reason a raw error score is not comparable between cameras (AUC micro vs per-video).

In [ ]:
cams = sorted(videos.camera.unique())
cols = 5; rows_n = int(np.ceil(len(cams) / cols))
fig, axes = plt.subplots(rows_n, cols, figsize=(14, 2.3 * rows_n))
for ax in axes.ravel():
    ax.axis("off")
for ax, c in zip(axes.ravel(), cams):
    v = videos[(videos.camera == c)].iloc[0].video
    f = list_frames(root / v)
    ax.imshow(cv2.cvtColor(cv2.imread(str(f[len(f) // 2])), cv2.COLOR_BGR2RGB)); ax.set_title(f"camera {c} ({v})", fontsize=9)
plt.tight_layout(); plt.savefig(FIG / "eda_cameras.png", bbox_inches="tight"); plt.show()

## Take-aways
- 330 normal training videos, 107 test videos; the test set is about 42% abnormal frames, so AUC (not accuracy) is the metric.
- Cameras differ a lot in number of videos, viewpoint and crowd: per-video / per-camera normalisation matters.
- Anomalies last many frames: temporal smoothing of the score is justified.